# 4. 기초통계와 회귀

웹 강의자료: [https://statjeong.github.io/moj-python-lecture/04_statistics.html](https://statjeong.github.io/moj-python-lecture/04_statistics.html)

이 장에서는 기술통계, 교차표, 두 집단 평균 비교, 선형회귀, 로지스틱 회귀를 실행해 봅니다. 실습 데이터가 10명뿐이라 결과의 통계적 의미보다는 코드가 어떻게 돌아가고 결과표를 어디서 읽는지에 집중합니다.

## 4.1 준비

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf

주소 = "https://raw.githubusercontent.com/StatJeong/moj-python-lecture/main/data/"
수용자 = pd.read_csv(주소 + "inmates.csv")
수용자["징벌여부"] = (수용자["징벌횟수"] > 0).astype(int)

## 4.2 기술통계

In [ ]:
수용자[["나이", "형기_개월", "상담횟수", "징벌횟수"]].describe().round(2)

| 행 | 뜻 |
|---|---|
| count | 값이 있는 칸 수 (상담횟수는 결측 1개로 9) |
| mean, std | 평균, 표준편차 |
| min, max | 최솟값, 최댓값 |
| 25%, 50%, 75% | 사분위수 (50%가 중앙값) |

평균과 중앙값을 따로 구할 수도 있습니다.

In [ ]:
print(수용자["나이"].mean())
print(수용자["나이"].median())

## 4.3 교차표

범주형 변수 두 개의 관계는 교차표로 봅니다.

In [ ]:
pd.crosstab(수용자["직업훈련"], 수용자["징벌여부"], margins=True)

행마다의 비율로 보려면 `normalize="index"`를 씁니다.

In [ ]:
pd.crosstab(수용자["직업훈련"], 수용자["징벌여부"], normalize="index").round(2)

## 4.4 두 집단 평균 비교 (t-검정)

직업훈련 참여자와 비참여자의 평균 징벌횟수를 비교합니다.

In [ ]:
참여 = 수용자[수용자["직업훈련"] == "Y"]["징벌횟수"]
비참여 = 수용자[수용자["직업훈련"] == "N"]["징벌횟수"]

print("참여 평균:", 참여.mean())
print("비참여 평균:", 비참여.mean())

결과 = stats.ttest_ind(참여, 비참여, equal_var=False)
print(결과)

`pvalue`가 0.05보다 작으면 두 집단 평균이 다르다고 판단하는 것이 일반적입니다. 다만 집단마다 5명이라 이 결과는 연습용으로만 봅니다.

## 4.5 선형회귀

상담횟수가 많을수록 징벌횟수가 어떻게 달라지는지 봅니다. `"결과 ~ 설명변수"` 형태의 식으로 모형을 씁니다.

In [ ]:
모형1 = smf.ols("징벌횟수 ~ 상담횟수", data=수용자).fit()
print(모형1.summary())

결과표에서 볼 곳은 다음과 같습니다.

| 위치 | 뜻 |
|---|---|
| `No. Observations` | 분석에 쓰인 관측치 수 |
| `R-squared` | 모형이 설명하는 변동의 비율 |
| `coef` | 계수. 설명변수가 1 늘 때 결과가 평균적으로 변하는 양 |
| P>\|t\| | 계수가 0인지 검정한 p값 |

관측치 수가 10이 아니라 9입니다. 상담횟수가 비어 있는 1007번이 자동으로 빠졌기 때문입니다. 계수만 따로 꺼낼 수도 있습니다.

In [ ]:
print(모형1.nobs)
print(모형1.params)

상담횟수의 계수가 음수이므로, 이 데이터에서는 상담횟수가 1회 많을수록 징벌횟수가 평균 약 0.42회 적습니다. 이것은 함께 움직이는 관계(연관)이고, 상담이 징벌을 줄였다는 인과관계를 뜻하지는 않습니다.

## 4.6 로지스틱 회귀

결과가 0 / 1인 경우(징벌을 받았는지 여부)에는 로지스틱 회귀를 씁니다.

In [ ]:
모형2 = smf.logit("징벌여부 ~ 나이", data=수용자).fit()
print(모형2.summary())

로지스틱 회귀의 계수는 지수함수(`np.exp`)를 씌워 오즈비로 읽습니다.

In [ ]:
np.exp(모형2.params)

나이의 오즈비가 약 0.83이므로, 나이가 1세 많을수록 징벌을 받을 오즈가 약 17% 낮습니다.

## 4.7 결과를 믿으면 안 되는 경우

이번에는 직업훈련 참여 여부로 징벌여부를 설명해 봅니다.

In [ ]:
모형3 = smf.logit("징벌여부 ~ 직업훈련", data=수용자).fit()
print(모형3.params)
print(모형3.bse)

코드는 끝까지 실행되지만 경고 메시지가 나오고, 계수가 20을 넘고 표준오차(`bse`)가 수만 단위로 나옵니다. 4.3절의 교차표를 다시 보면 직업훈련 N인 5명이 모두 징벌을 받았습니다. 이렇게 한 집단의 결과가 전부 같으면 계수를 추정할 수 없습니다(완전 분리).

오류 없이 실행됐다고 해서 결과가 맞는 것은 아닙니다. 경고 메시지와 비정상적으로 큰 값은 확인하고 넘어가야 합니다. 5장에서 AI가 준 코드를 쓸 때도 같은 원칙이 적용됩니다.

## 4.8 연습문제

1. 시설별 나이의 평균과 표준편차를 구하세요.

In [ ]:
# 여기에 코드를 작성해 보세요

2. 징벌횟수를 나이로 설명하는 선형회귀를 실행하고 나이의 계수를 확인하세요.

In [ ]:
# 여기에 코드를 작성해 보세요